# Pre-presentation data checks

Run this notebook **before** presenting the Executive Sales Dashboard.

It validates:
1. `data.xlsx` grain, quality, and sheet alignment
2. `pulse_cache.json` structure and consistency with Excel
3. Business sanity checks useful for a presentation

Working directory should be the project folder: `sales-executive-dashboard/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

BASE = Path('.').resolve()
XLSX = BASE / 'data.xlsx'
CACHE = BASE / 'pulse_cache.json'

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Project folder:', BASE)
print('data.xlsx exists:', XLSX.exists(), '| size MB:', round(XLSX.stat().st_size/1e6, 2) if XLSX.exists() else None)
print('pulse_cache.json exists:', CACHE.exists(), '| size MB:', round(CACHE.stat().st_size/1e6, 2) if CACHE.exists() else None)

## 1. Load Excel sheets

In [ ]:
xl = pd.ExcelFile(XLSX)
print('Sheets:', xl.sheet_names)

sales = xl.parse('Sales_Data')
targets = xl.parse('Monthly_Targets')

print('\nSales_Data shape:', sales.shape)
print('Monthly_Targets shape:', targets.shape)
print('\nSales columns:')
print(sales.dtypes)
print('\nTarget columns:')
print(targets.dtypes)

sales.head(3)

## 2. Grain checks (most important before joining)

| Table | Expected grain |
|---|---|
| Sales_Data | 1 row = 1 order line (`Order_ID` + `Order_Line`) |
| Monthly_Targets | 1 row = 1 sales rep × month |

In [ ]:
print('=== Sales_Data grain ===')
print('rows:', len(sales))
print('unique Order_ID:', sales['Order_ID'].nunique())
print('avg lines per order:', round(len(sales) / sales['Order_ID'].nunique(), 2))

line_dups = sales.duplicated(subset=['Order_ID', 'Order_Line']).sum()
print('duplicate Order_ID + Order_Line:', line_dups)
assert line_dups == 0, 'Sales grain is broken: Order_ID + Order_Line should be unique'

multi_region = (sales.groupby('Sales_Rep_ID')['Region'].nunique() != 1).sum()
print('reps with >1 region:', int(multi_region))
assert multi_region == 0, 'Each Sales_Rep_ID must belong to exactly one Region'

print('\n=== Monthly_Targets grain ===')
print('rows:', len(targets))
print('unique Sales_Rep_ID:', targets['Sales_Rep_ID'].nunique())
tgt_dups = targets.duplicated(subset=['Sales_Rep_ID', 'Month_Start']).sum()
print('duplicate Sales_Rep_ID + Month_Start:', tgt_dups)
assert tgt_dups == 0, 'Target grain is broken: Sales_Rep_ID + Month_Start should be unique'

n_reps = targets['Sales_Rep_ID'].nunique()
n_months = targets['Month_Start'].nunique()
print(f'reps × months = {n_reps} × {n_months} = {n_reps * n_months} (actual rows {len(targets)})')
assert len(targets) == n_reps * n_months, 'Targets should be a full rep × month grid'

print('\nGrain checks passed.')

## 3. Data quality — nulls, flags, ranges

In [ ]:
print('=== Null counts (Sales_Data) ===')
nulls_sales = sales.isna().sum()
print(nulls_sales[nulls_sales > 0] if nulls_sales.any() else 'No nulls')

print('\n=== Null counts (Monthly_Targets) ===')
nulls_tgt = targets.isna().sum()
print(nulls_tgt[nulls_tgt > 0] if nulls_tgt.any() else 'No nulls')

print('\n=== Flag value checks ===')
print('On_Time_Delivery_Flag:', sorted(sales['On_Time_Delivery_Flag'].dropna().unique().tolist()))
print('Returned_Flag:', sorted(sales['Returned_Flag'].dropna().unique().tolist()))

print('\n=== Numeric ranges (Sales_Data) ===')
num_cols = ['Net_Sales_INR', 'Gross_Profit_INR', 'Units', 'Discount_Pct', 'Return_Amount_INR', 'NPS_Score']
display(sales[num_cols].describe())

print('Negative Net_Sales rows:', int((sales['Net_Sales_INR'] < 0).sum()))
print('Discount outside 0-1:', int(((sales['Discount_Pct'] < 0) | (sales['Discount_Pct'] > 1)).sum()))
print('NPS outside 0-10:', int(((sales['NPS_Score'] < 0) | (sales['NPS_Score'] > 10)).sum()))

print('\n=== Date coverage ===')
print('Sales Order_Date:', sales['Order_Date'].min(), '->', sales['Order_Date'].max())
print('Targets Month_Start:', targets['Month_Start'].min(), '->', targets['Month_Start'].max())

## 4. Sheet alignment — same reps, no orphan targets

In [ ]:
sales_reps = set(sales['Sales_Rep_ID'].unique())
tgt_reps = set(targets['Sales_Rep_ID'].unique())

print('Reps in sales:', len(sales_reps))
print('Reps in targets:', len(tgt_reps))
print('In sales only:', sorted(sales_reps - tgt_reps)[:10], '...' if len(sales_reps - tgt_reps) > 10 else '')
print('In targets only:', sorted(tgt_reps - sales_reps)[:10], '...' if len(tgt_reps - sales_reps) > 10 else '')
assert sales_reps == tgt_reps, 'Sales and Targets rep sets should match'

print('\nRegions:', sorted(sales['Region'].unique()))
print('Segments:', sorted(sales['Segment'].unique()))
print('Channels:', sorted(sales['Channel'].unique()))
print('Categories:', sorted(sales['Product_Category'].unique()))
print('\nAlignment checks passed.')

## 5. Why we do **not** join targets onto every sales line

If you naively merge targets onto order lines, each monthly target is counted once per line and attainment becomes wrong.

In [ ]:
sales['Month_Start'] = sales['Order_Date'].values.astype('datetime64[M]')

# Correct: aggregate sales to rep × month, then join targets
actual = (
    sales.groupby(['Sales_Rep_ID', 'Month_Start'], as_index=False)
    .agg(Actual_Net_Sales=('Net_Sales_INR', 'sum'))
)
correct = actual.merge(
    targets[['Sales_Rep_ID', 'Month_Start', 'Monthly_Sales_Target_INR']],
    on=['Sales_Rep_ID', 'Month_Start'],
    how='left'
)
correct_attain = correct['Actual_Net_Sales'].sum() / correct['Monthly_Sales_Target_INR'].sum()

# Wrong: join target onto every order line, then sum
wrong = sales.merge(
    targets[['Sales_Rep_ID', 'Month_Start', 'Monthly_Sales_Target_INR']],
    on=['Sales_Rep_ID', 'Month_Start'],
    how='left'
)
wrong_attain = wrong['Net_Sales_INR'].sum() / wrong['Monthly_Sales_Target_INR'].sum()

print('Correct attainment (targets summed once per rep-month):', f'{correct_attain:.1%}')
print('Wrong attainment (targets duplicated on every line):   ', f'{wrong_attain:.1%}')
print('Inflation factor on target sum:', round(wrong['Monthly_Sales_Target_INR'].sum() / correct['Monthly_Sales_Target_INR'].sum(), 2))

## 6. Load `pulse_cache.json` and compare with Excel

In [ ]:
pulse = json.loads(CACHE.read_text(encoding='utf-8'))
D, dims, T = pulse['D'], pulse['dims'], pulse['T']

print('Top-level keys:', list(pulse.keys()))
print('D (sales columnar) keys:', list(D.keys()))
print('dims keys:', list(dims.keys()))
print()
print('Excel sales rows:', len(sales), '| cache D rows:', len(D['ns']))
print('Excel target rows:', len(targets), '| cache T rows:', len(T))

assert len(D['ns']) == len(sales), 'Cache sales row count mismatch'
assert len(T) == len(targets), 'Cache target row count mismatch'

# Totals should match (cache stores rounded INR integers)
excel_ns = sales['Net_Sales_INR'].sum()
cache_ns = sum(D['ns'])
excel_gp = sales['Gross_Profit_INR'].sum()
cache_gp = sum(D['gp'])

print('\nNet Sales Excel vs cache:', f'{excel_ns:,.0f}', 'vs', f'{cache_ns:,.0f}',
      '| abs diff', f'{abs(excel_ns - cache_ns):,.0f}')
print('Gross Profit Excel vs cache:', f'{excel_gp:,.0f}', 'vs', f'{cache_gp:,.0f}',
      '| abs diff', f'{abs(excel_gp - cache_gp):,.0f}')

# Rounding can create small INR diffs; flag only material gaps
assert abs(excel_ns - cache_ns) < 20_000, 'Net Sales mismatch looks too large'
assert abs(excel_gp - cache_gp) < 20_000, 'Gross Profit mismatch looks too large'

print('\nCache vs Excel consistency checks passed.')

In [ ]:
# Dimension labels in cache should match Excel uniques
checks = {
    'reg': 'Region',
    'seg': 'Segment',
    'ch': 'Channel',
    'cat': 'Product_Category',
    'rep': 'Sales_Rep_ID',
}
for key, col in checks.items():
    excel_vals = sorted(sales[col].unique().tolist())
    cache_vals = dims[key]
    ok = excel_vals == cache_vals
    print(f'{key:4} ({col}): match={ok} | n={len(cache_vals)}')
    assert ok, f'Dimension mismatch for {col}'

print('\nDimension label checks passed.')

## 7. Presentation-ready insights (from Excel)

These are live numbers you can quote. Re-run after updating `data.xlsx`.

In [ ]:
def crore(x):
    return x / 1e7

sales['Year'] = sales['Order_Date'].dt.year
by_year = sales.groupby('Year').agg(
    Net_Sales=('Net_Sales_INR', 'sum'),
    Gross_Profit=('Gross_Profit_INR', 'sum'),
    Units=('Units', 'sum'),
    Lines=('Order_ID', 'count'),
)
by_year['Gross_Margin'] = by_year['Gross_Profit'] / by_year['Net_Sales']
by_year['Net_Sales_Cr'] = crore(by_year['Net_Sales'])
display(by_year)

if {2024, 2025}.issubset(by_year.index):
    growth = by_year.loc[2025, 'Net_Sales'] / by_year.loc[2024, 'Net_Sales'] - 1
    print(f'YoY Net Sales growth (2025 vs 2024): {growth:.1%}')

In [ ]:
# Attainment at correct grain
actual_ym = (
    sales.groupby(['Sales_Rep_ID', 'Month_Start'], as_index=False)
    .agg(Actual=('Net_Sales_INR', 'sum'))
)
att = actual_ym.merge(
    targets[['Sales_Rep_ID', 'Month_Start', 'Monthly_Sales_Target_INR']],
    on=['Sales_Rep_ID', 'Month_Start'], how='outer'
).fillna(0)
att['hit'] = att['Actual'] >= att['Monthly_Sales_Target_INR']

overall = att['Actual'].sum() / att['Monthly_Sales_Target_INR'].sum()
hit_rate = att['hit'].mean()
print(f'Overall Net Sales attainment: {overall:.1%}')
print(f'Rep-months at or above target: {hit_rate:.1%} ({att["hit"].sum()} of {len(att)})')

In [ ]:
print('=== By Region ===')
reg = sales.groupby('Region').agg(
    Net_Sales=('Net_Sales_INR', 'sum'),
    Gross_Profit=('Gross_Profit_INR', 'sum'),
    On_Time=('On_Time_Delivery_Flag', lambda s: (s == 'Yes').mean()),
    Return_Rate=('Return_Amount_INR', 'sum'),
)
reg['Share'] = reg['Net_Sales'] / reg['Net_Sales'].sum()
reg['Margin'] = reg['Gross_Profit'] / reg['Net_Sales']
reg['Return_Rate'] = reg['Return_Rate'] / reg['Net_Sales']
reg['Net_Sales_Cr'] = crore(reg['Net_Sales'])
display(reg.sort_values('Net_Sales', ascending=False)[['Net_Sales_Cr', 'Share', 'Margin', 'On_Time', 'Return_Rate']])

print('\n=== By Category ===')
cat = sales.groupby('Product_Category').agg(
    Net_Sales=('Net_Sales_INR', 'sum'),
    Gross_Profit=('Gross_Profit_INR', 'sum'),
)
cat['Share'] = cat['Net_Sales'] / cat['Net_Sales'].sum()
cat['Margin'] = cat['Gross_Profit'] / cat['Net_Sales']
cat['Net_Sales_Cr'] = crore(cat['Net_Sales'])
display(cat.sort_values('Net_Sales', ascending=False))

print('\n=== By Channel ===')
ch = sales.groupby('Channel').agg(
    Net_Sales=('Net_Sales_INR', 'sum'),
    Avg_Discount=('Discount_Pct', 'mean'),
    Margin=('Gross_Profit_INR', lambda s: s.sum() / sales.loc[s.index, 'Net_Sales_INR'].sum()),
    On_Time=('On_Time_Delivery_Flag', lambda s: (s == 'Yes').mean()),
    Return_Rate=('Return_Amount_INR', lambda s: s.sum() / sales.loc[s.index, 'Net_Sales_INR'].sum()),
)
ch['Net_Sales_Cr'] = crore(ch['Net_Sales'])
display(ch.sort_values('Net_Sales', ascending=False))

In [ ]:
# Concentration / outlier style checks
cust = sales.groupby('Customer_Name')['Net_Sales_INR'].sum().sort_values(ascending=False)
top10_share = cust.head(10).sum() / cust.sum()
print(f'Top 10 customers share of Net Sales: {top10_share:.1%}')

rep_tot = sales.groupby('Sales_Rep_ID')['Net_Sales_INR'].sum()
print(f'Rep Net Sales range: ₹{crore(rep_tot.min()):.1f} Cr  ->  ₹{crore(rep_tot.max()):.1f} Cr')
print(f'Rep ratio max/min: {rep_tot.max() / rep_tot.min():.1f}x')

# Seasonality
sales['Month'] = sales['Order_Date'].dt.month
q4_share = sales.loc[sales['Month'].isin([10, 11, 12]), 'Net_Sales_INR'].sum() / sales['Net_Sales_INR'].sum()
print(f'Q4 (Oct-Dec) share of all Net Sales: {q4_share:.1%}')

# Low-margin hardware lines (presentation risk story)
hw = sales[sales['Product_Category'] == 'Hardware']
hw_margin = hw['Gross_Profit_INR'] / hw['Net_Sales_INR']
low_m = ((hw_margin < 0.10) & hw['Net_Sales_INR'].gt(0)).sum()
print(f'Hardware lines with margin < 10%: {low_m}')

## 8. Final go / no-go checklist

In [ ]:
checklist = {
    'data.xlsx present': XLSX.exists(),
    'pulse_cache.json present': CACHE.exists(),
    'Sales grain unique (Order_ID + Order_Line)': sales.duplicated(subset=['Order_ID', 'Order_Line']).sum() == 0,
    'Target grain unique (Rep + Month)': targets.duplicated(subset=['Sales_Rep_ID', 'Month_Start']).sum() == 0,
    'Sales/Target rep sets match': set(sales['Sales_Rep_ID']) == set(targets['Sales_Rep_ID']),
    'Each rep has one region': (sales.groupby('Sales_Rep_ID')['Region'].nunique() == 1).all(),
    'Cache sales rows == Excel': len(D['ns']) == len(sales),
    'Cache target rows == Excel': len(T) == len(targets),
    'No nulls in key sales metrics': sales[['Net_Sales_INR', 'Gross_Profit_INR', 'Units']].isna().sum().sum() == 0,
    'Cache newer or equal to Excel': CACHE.stat().st_mtime >= XLSX.stat().st_mtime,
}

for item, ok in checklist.items():
    print(('PASS' if ok else 'FAIL'), '-', item)

if all(checklist.values()):
    print('\nReady to present.')
else:
    print('\nFix FAIL items before presenting.')
    if not checklist['Cache newer or equal to Excel']:
        print('Tip: run  python Pulse_Builder.py  or use admin Rebuild data')